<div style="background-color:#F3F2EE">
    <br /><br />
        <p style="text-align: center;">
            <font size="6" color='#0A1781'>
                <strong>
                    Databricks Certification Learning Knowledge Graph
                </strong>
              </font>
        </p>
        <p style="text-align: center;">
            <font size="6" color='#C58A1E'>
                <strong>
                    Neo4j Load and Cypher Queries
                </strong>
            </font>
        </p>
        <p style="text-align: center;">
            <font size="5" color='#C58A1E'>
                <strong>
                    Iniciar a carga do Knowledge Graph no Neo4j e executar as primeiras consultas Cypher.
                </strong>
            </font>
        </p>
    <br />
</div>

<div style="background-color:#F3F2EE">
    <p style="text-align: right;">
      <font size="4" color='#444444'>
            Roberto SSoares - LfLngLrnng
      </font>
    </p>
    <p style="text-align: right;"><font size="2" color='#444444'>
        <a href="https://www.linkedin.com/in/roberto-dos-santos-soares/">in/roberto-dos-santos-soares</a><br /><a href="https://roberto-ssoares.github.io/meu-portfolio/">Portifólio: roberto-ssoares</a>
    </p>
    <p style="text-align: right;">
        <font size="4" color='#444444'>
            " [+] Faturamento [-] Custo [+] Qualidade de vida "
        </font>
        <br />
        <font size="2" color='#918e8e'>"Mestre Bruno Jardim"
        </font>
    </p>        
    <p style="text-align: right;">        
        <font size="2" color='#918e8e'>           
        </font>
    </p>
</div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>📌 Objetivo</strong></font>

<font size="2" color='#66666'>

>- **Ações realizadas**
    - ***Configuração de conexão com Neo4j.***
    - ***Leitura dos arquivos `nodes_all.csv` e `edges_all.csv`.***
    - ***Criação de constraint para identificação única dos nós.***
    - ***Carga dos nós e relacionamentos no banco de grafos.***
    - ***Execução de consultas exploratórias em Cypher.***

>- **Justificativa técnica**
    - Após validar o grafo em Python com NetworkX e PyVis, o próximo passo é persistir o Knowledge Graph em uma base especializada em grafos,
    - permitindo consultas semânticas, navegação por relações, expansão incremental e uso futuro em aplicações analíticas.

>- **Resultados esperados**
    - Ao final deste notebook, teremos o Knowledge Graph carregado no Neo4j e validado por consultas Cypher iniciais.

---

</font></div>

In [ ]:
#!uv pip install watermark -q -U
#!uv pip install tabulate -q -U
#!uv pip install networkx pyvis matplotlib -q -U
#!uv pip install dotenv
#!uv pip install neo4j -q

In [1]:
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [2]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "RobertoSSoares-LfLngLrnng"

Author: RobertoSSoares-LfLngLrnng



In [3]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [4]:
PROJECT_NAME = "Databricks Certification Learning Knowledge Graph"

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORTS_DIR = DATA_DIR / "exports"
DOCS_DIR = BASE_DIR / "docs"
CYPHER_DIR = BASE_DIR / "cypher"

for directory in [PROCESSED_DIR, EXPORTS_DIR, DOCS_DIR, CYPHER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BASE_DIR

WindowsPath('D:/_DS-Projects/Data-Science/databricks-learning-kg')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 1. Carregar .env</strong></font>

<font size="2" color='#66666'></font></div>

In [5]:
load_dotenv(BASE_DIR / ".env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "DigitalTwins-KG")

{
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_USER": NEO4J_USER,
    "NEO4J_DATABASE": NEO4J_DATABASE,
    "password_loaded": bool(NEO4J_PASSWORD),
}

{'NEO4J_URI': 'neo4j://localhost:7687',
 'NEO4J_USER': 'neo4j',
 'NEO4J_DATABASE': 'digitaltwinskg',
 'password_loaded': True}

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 2. Testar conexão</strong></font>

<font size="2" color='#66666'></font></div>

In [6]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("RETURN 'Neo4j connection OK' AS status")
    print(result.single()["status"])

Neo4j connection OK


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 3. Carregar CSVs</strong></font>

<font size="2" color='#66666'></font></div>

In [7]:
nodes_path = PROCESSED_DIR / "nodes_all.csv"
edges_path = PROCESSED_DIR / "edges_all.csv"

nodes_all = pd.read_csv(nodes_path).fillna("")
edges_all = pd.read_csv(edges_path).fillna("")

display(nodes_all.head())
display(edges_all.head())

print(f"Total de nós: {len(nodes_all)}")
print(f"Total de relacionamentos: {len(edges_all)}")

,node_id,node_label,name,category,description,status,priority
0,CERT_DEA_001,Certification,Databricks Certified Data Engineer Associate,Associate,Certificação alvo para consolidar fundamentos de Engenharia de Dados na plataforma Databricks.,planned,
1,D01,ExamDomain,Databricks Intelligence Platform,medium,"Fundamentos da plataforma, workspace, compute e arquitetura Lakehouse.",,medium
2,D02,ExamDomain,Development and Ingestion,high,"Desenvolvimento, ingestão de dados e uso de SQL/PySpark.",,high
3,D03,ExamDomain,Data Processing & Transformations,high,"Transformações, Delta Lake, tabelas, joins, agregações e arquitetura medalhão.",,high
4,D04,ExamDomain,Productionizing Data Pipelines,high,"Jobs, Workflows, tarefas, agendamento, monitoramento e operação.",,high


,edge_id,source_id,source_label,relationship,target_id,target_label,weight,description
0,E0001,CERT_DEA_001,Certification,HAS_DOMAIN,D01,ExamDomain,0.10,A certificação cobre o domínio Databricks Intelligence Platform.
1,E0002,CERT_DEA_001,Certification,HAS_DOMAIN,D02,ExamDomain,0.30,A certificação cobre o domínio Development and Ingestion.
2,E0003,CERT_DEA_001,Certification,HAS_DOMAIN,D03,ExamDomain,0.31,A certificação cobre o domínio Data Processing & Transformations.
3,E0004,CERT_DEA_001,Certification,HAS_DOMAIN,D04,ExamDomain,0.18,A certificação cobre o domínio Productionizing Data Pipelines.
4,E0005,CERT_DEA_001,Certification,HAS_DOMAIN,D05,ExamDomain,0.11,A certificação cobre o domínio Data Governance & Quality.


Total de nós: 93
Total de relacionamentos: 172


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 4. Funções auxiliares</strong></font>

<font size="2" color='#66666'></font></div>

In [8]:
def safe_neo4j_name(value: str) -> str:
    """
    Garante que labels e tipos de relacionamento sejam seguros para composição em Cypher.
    Neo4j não permite parametrizar labels e relationship types diretamente.
    """
    value = str(value).strip()
    value = re.sub(r"[^A-Za-z0-9_]", "_", value)
    
    if not value:
        raise ValueError("Nome inválido para Neo4j.")
    
    if value[0].isdigit():
        value = f"_{value}"
    
    return value


def row_to_props(row: pd.Series, exclude: set[str]) -> dict:
    """
    Converte uma linha de DataFrame em dicionário de propriedades.
    Remove colunas de controle e valores vazios quando adequado.
    """
    props = {}
    
    for key, value in row.items():
        if key in exclude:
            continue
        
        if pd.isna(value):
            continue
        
        if value == "":
            props[key] = ""
        else:
            props[key] = value
    
    return props

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 5. Criar constraint</strong></font>

<font size="2" color='#66666'></font></div>

In [9]:
constraint_cypher = """
CREATE CONSTRAINT kg_node_id_unique IF NOT EXISTS
FOR (n:KGNode)
REQUIRE n.node_id IS UNIQUE
"""

with driver.session(database=NEO4J_DATABASE) as session:
    session.run(constraint_cypher)

print("Constraint criada/verificada com sucesso.")

Constraint criada/verificada com sucesso.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 6. Limpar carga anterior do projeto</strong></font>

<font size="2" color='#66666'>

- Use esta célula apenas porque estamos em ambiente de desenvolvimento.    

</font></div>

In [11]:
delete_cypher = """
MATCH (n:KGNode)
DETACH DELETE n
"""

with driver.session(database=NEO4J_DATABASE) as session:
    session.run(delete_cypher)

print("Carga anterior removida com sucesso.")

Carga anterior removida com sucesso.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 7. Carregar nós</strong></font>

<font size="2" color='#66666'></font></div>

In [12]:
def load_node(tx, node: dict):
    node_label = safe_neo4j_name(node["node_label"])
    
    cypher = f"""
    MERGE (n:KGNode:{node_label} {{node_id: $node_id}})
    SET n += $props
    """
    
    tx.run(
        cypher,
        node_id=str(node["node_id"]),
        props=node["props"],
    )


node_records = []

for _, row in nodes_all.iterrows():
    props = row_to_props(row, exclude={"node_id"})
    node_records.append(
        {
            "node_id": str(row["node_id"]),
            "node_label": str(row["node_label"]),
            "props": props,
        }
    )

with driver.session(database=NEO4J_DATABASE) as session:
    for node in node_records:
        session.execute_write(load_node, node)

print(f"Nós carregados: {len(node_records)}")

Nós carregados: 93


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 8. Carregar relacionamentos</strong></font>

<font size="2" color='#66666'></font></div>

In [13]:
def load_relationship(tx, edge: dict):
    relationship = safe_neo4j_name(edge["relationship"])
    
    cypher = f"""
    MATCH (s:KGNode {{node_id: $source_id}})
    MATCH (t:KGNode {{node_id: $target_id}})
    MERGE (s)-[r:{relationship} {{edge_id: $edge_id}}]->(t)
    SET r += $props
    """
    
    tx.run(
        cypher,
        source_id=str(edge["source_id"]),
        target_id=str(edge["target_id"]),
        edge_id=str(edge["edge_id"]),
        props=edge["props"],
    )


edge_records = []

for _, row in edges_all.iterrows():
    props = row_to_props(row, exclude={"source_id", "target_id"})
    edge_records.append(
        {
            "edge_id": str(row["edge_id"]),
            "source_id": str(row["source_id"]),
            "target_id": str(row["target_id"]),
            "relationship": str(row["relationship"]),
            "props": props,
        }
    )

with driver.session(database=NEO4J_DATABASE) as session:
    for edge in edge_records:
        session.execute_write(load_relationship, edge)

print(f"Relacionamentos carregados: {len(edge_records)}")


Relacionamentos carregados: 172


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 9. Validar quantidade carregada</strong></font>

<font size="2" color='#66666'></font></div>

In [14]:
validation_queries = {
    "nodes": "MATCH (n:KGNode) RETURN count(n) AS total",
    "relationships": "MATCH (:KGNode)-[r]->(:KGNode) RETURN count(r) AS total",
}

validation_results = []

with driver.session(database=NEO4J_DATABASE) as session:
    for check_name, query in validation_queries.items():
        result = session.run(query)
        total = result.single()["total"]
        validation_results.append(
            {
                "check": check_name,
                "total_in_neo4j": total,
            }
        )

validation_df = pd.DataFrame(validation_results)

validation_df

,check,total_in_neo4j
0,nodes,93
1,relationships,172


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 10. Nós por tipo</strong></font>

<font size="2" color='#66666'></font></div>

In [15]:
query = """
MATCH (n:KGNode)
RETURN n.node_label AS node_label, count(n) AS total
ORDER BY total DESC
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query)
    nodes_by_label_neo4j = pd.DataFrame([dict(record) for record in records])

nodes_by_label_neo4j

,node_label,total
0,Topic,46
1,Subtopic,20
2,Skill,6
3,ExamDomain,5
4,Notebook,5
5,Resource,4
6,Lab,4
7,StudySession,1
8,Snapshot,1
9,Certification,1


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 11. Relacionamentos por tipo</strong></font>

<font size="2" color='#66666'></font></div>

In [16]:
query = """
MATCH (:KGNode)-[r]->(:KGNode)
RETURN type(r) AS relationship, count(r) AS total
ORDER BY total DESC
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query)
    relationships_by_type_neo4j = pd.DataFrame([dict(record) for record in records])

relationships_by_type_neo4j

,relationship,total
0,COVERS,46
1,CAPTURES_STATUS_OF,46
2,HAS_SUBTOPIC,20
3,PREREQUISITE_FOR,15
4,SUPPORTED_BY,11
5,TEACHES,10
6,PRACTICES,8
7,IMPLEMENTS,7
8,HAS_DOMAIN,5
9,STUDIED,4


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 12. Tópicos por domínio</strong></font>

<font size="2" color='#66666'></font></div>

In [17]:
query = """
MATCH (d:ExamDomain)-[:COVERS]->(t:Topic)
RETURN
    d.name AS domain,
    count(t) AS total_topics
ORDER BY total_topics DESC
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query)
    topics_by_domain = pd.DataFrame([dict(record) for record in records])

topics_by_domain

,domain,total_topics
0,Data Processing & Transformations,13
1,Development and Ingestion,9
2,Productionizing Data Pipelines,9
3,Data Governance & Quality,9
4,Databricks Intelligence Platform,6


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 13. Tópicos de maior prioridade</strong></font>

<font size="2" color='#66666'></font></div>

In [18]:
query = """
MATCH (t:Topic)
WHERE t.priority = 'high'
RETURN
    t.node_id AS topic_id,
    t.name AS topic,
    t.category AS category,
    t.status AS status,
    t.priority AS priority
ORDER BY t.category, t.name
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query)
    high_priority_topics = pd.DataFrame([dict(record) for record in records])

high_priority_topics.head(20)

,topic_id,topic,category,status,priority
0,T015,PySpark DataFrame API,Development,initial_mapping,high
1,T014,Spark SQL,Development,initial_mapping,high
2,T045,Data Quality,Governance,initial_mapping,high
3,T043,Permissions,Governance,not_started,high
4,T038,Unity Catalog,Governance,not_started,high
5,T007,Data Ingestion,Ingestion,initial_mapping,high
6,T011,File Formats,Ingestion,initial_mapping,high
7,T013,Schema Definition,Ingestion,initial_mapping,high
8,T012,Schema Inference,Ingestion,initial_mapping,high
9,T008,read_files,Ingestion,not_started,high


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 14. Tópicos ainda sem prática de notebook</strong></font>

<font size="2" color='#66666'></font></div>

In [19]:
query = """
MATCH (t:Topic)
WHERE NOT EXISTS {
    MATCH (:Notebook)-[:PRACTICES]->(t)
}
RETURN
    t.node_id AS topic_id,
    t.name AS topic,
    t.category AS category,
    t.priority AS priority,
    t.status AS status
ORDER BY t.priority DESC, t.category, t.name
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query)
    topics_without_notebook = pd.DataFrame([dict(record) for record in records])

topics_without_notebook.head(20)

,topic_id,topic,category,priority,status
0,T039,Catalog,Governance,medium,not_started
1,T044,Lineage,Governance,medium,not_started
2,T046,Naming Conventions,Governance,medium,initial_mapping
3,T040,Schema,Governance,medium,initial_mapping
4,T041,Table,Governance,medium,initial_mapping
5,T042,Volume,Governance,medium,not_started
6,T010,Auto Loader,Ingestion,medium,not_started
7,T009,COPY INTO,Ingestion,medium,not_started
8,T003,Compute,Platform,medium,not_started
9,T002,Databricks Workspace,Platform,medium,not_started


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>📌 Observe:</strong></font>

<font size="2" color='#66666'>

>- Essa consulta é uma das mais importantes do projeto, porque já mostra gaps de evidência prática.

---

</font></div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 15. Pré-requisitos mais estruturantes</strong></font>

<font size="2" color='#66666'></font></div>

In [20]:
query = """
MATCH (t:Topic)-[:PREREQUISITE_FOR]->(next:Topic)
RETURN
    t.node_id AS topic_id,
    t.name AS topic,
    count(next) AS unlocks_topics
ORDER BY unlocks_topics DESC, topic
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query)
    prerequisite_hubs = pd.DataFrame([dict(record) for record in records])

prerequisite_hubs

,topic_id,topic,unlocks_topics
0,T007,Data Ingestion,3
1,T039,Catalog,1
2,T029,Databricks Workflows,1
3,T016,Delta Lake,1
4,T017,Delta Table,1
5,T030,Jobs,1
6,T023,Joins,1
7,T001,Lakehouse Architecture,1
8,T040,Schema,1
9,T012,Schema Inference,1


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 16. Caminho da certificação até tópicos</strong></font>

<font size="2" color='#66666'></font></div>

In [21]:
query = """
MATCH path = (:Certification)-[:HAS_DOMAIN]->(:ExamDomain)-[:COVERS]->(:Topic)
RETURN
    [node IN nodes(path) | node.name] AS learning_path
LIMIT 20
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query)
    certification_paths = pd.DataFrame([dict(record) for record in records])

certification_paths.head(20)

,learning_path
0,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Lakehouse Architecture]"
1,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Databricks Workspace]"
2,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Compute]"
3,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, SQL Warehouse]"
4,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Notebooks]"
5,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Repos]"
6,"[Databricks Certified Data Engineer Associate, Development and Ingestion, Data Ingestion]"
7,"[Databricks Certified Data Engineer Associate, Development and Ingestion, read_files]"
8,"[Databricks Certified Data Engineer Associate, Development and Ingestion, COPY INTO]"
9,"[Databricks Certified Data Engineer Associate, Development and Ingestion, Auto Loader]"


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 17. Caminho da certificação até tópicos</strong></font>

<font size="2" color='#66666'></font></div>

In [22]:
query = """
MATCH path = (:Certification)-[:HAS_DOMAIN]->(:ExamDomain)-[:COVERS]->(:Topic)
RETURN
    [node IN nodes(path) | node.name] AS learning_path
LIMIT 20
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query)
    certification_paths = pd.DataFrame([dict(record) for record in records])

certification_paths.head(20)

,learning_path
0,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Lakehouse Architecture]"
1,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Databricks Workspace]"
2,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Compute]"
3,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, SQL Warehouse]"
4,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Notebooks]"
5,"[Databricks Certified Data Engineer Associate, Databricks Intelligence Platform, Repos]"
6,"[Databricks Certified Data Engineer Associate, Development and Ingestion, Data Ingestion]"
7,"[Databricks Certified Data Engineer Associate, Development and Ingestion, read_files]"
8,"[Databricks Certified Data Engineer Associate, Development and Ingestion, COPY INTO]"
9,"[Databricks Certified Data Engineer Associate, Development and Ingestion, Auto Loader]"


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 18. Consulta para abrir no Neo4j Browser</strong></font>

<font size="2" color='#66666'></font></div>

In [23]:
browser_query = """
MATCH path = (c:Certification)-[:HAS_DOMAIN]->(d:ExamDomain)-[:COVERS]->(t:Topic)
RETURN path
LIMIT 100
"""

print(browser_query)


MATCH path = (c:Certification)-[:HAS_DOMAIN]->(d:ExamDomain)-[:COVERS]->(t:Topic)
RETURN path
LIMIT 100



<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 19. Consulta visual com notebooks e skills</strong></font>

<font size="2" color='#66666'></font></div>

In [24]:
browser_query_2 = """
MATCH path = (:Notebook)-[:PRACTICES]->(:Topic)<-[:SUPPORTED_BY]-(:Skill)
RETURN path
LIMIT 100
"""

print(browser_query_2)


MATCH path = (:Notebook)-[:PRACTICES]->(:Topic)<-[:SUPPORTED_BY]-(:Skill)
RETURN path
LIMIT 100



<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 20. Exportar resultados</strong></font>

<font size="2" color='#66666'></font></div>

In [25]:
topics_by_domain.to_csv(EXPORTS_DIR / "neo4j_topics_by_domain.csv", index=False, encoding="utf-8")
high_priority_topics.to_csv(EXPORTS_DIR / "neo4j_high_priority_topics.csv", index=False, encoding="utf-8")
topics_without_notebook.to_csv(EXPORTS_DIR / "neo4j_topics_without_notebook.csv", index=False, encoding="utf-8")
prerequisite_hubs.to_csv(EXPORTS_DIR / "neo4j_prerequisite_hubs.csv", index=False, encoding="utf-8")

print("Arquivos exportados com sucesso.")

Arquivos exportados com sucesso.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 21. Gerar aquivo Cypher de referência</strong></font>

<font size="2" color='#66666'></font></div>

In [26]:
cypher_reference = """// Cypher Reference — Databricks Learning KG

// 1. Visualizar certificação, domínios e tópicos
MATCH path = (c:Certification)-[:HAS_DOMAIN]->(d:ExamDomain)-[:COVERS]->(t:Topic)
RETURN path
LIMIT 100;

// 2. Tópicos por domínio
MATCH (d:ExamDomain)-[:COVERS]->(t:Topic)
RETURN d.name AS domain, count(t) AS total_topics
ORDER BY total_topics DESC;

// 3. Tópicos sem notebook prático
MATCH (t:Topic)
WHERE NOT EXISTS {
    MATCH (:Notebook)-[:PRACTICES]->(t)
}
RETURN t.node_id AS topic_id, t.name AS topic, t.category AS category, t.priority AS priority
ORDER BY t.priority DESC, t.category, t.name;

// 4. Pré-requisitos mais estruturantes
MATCH (t:Topic)-[:PREREQUISITE_FOR]->(next:Topic)
RETURN t.name AS topic, count(next) AS unlocks_topics
ORDER BY unlocks_topics DESC;

// 5. Notebooks conectados a skills
MATCH path = (:Notebook)-[:PRACTICES]->(:Topic)<-[:SUPPORTED_BY]-(:Skill)
RETURN path
LIMIT 100;

// 6. Snapshot inicial de progresso
MATCH path = (:Snapshot)-[:CAPTURES_STATUS_OF]->(:Topic)
RETURN path
LIMIT 100;
"""

cypher_path = CYPHER_DIR / "03_initial_kg_queries.cypher"
cypher_path.write_text(cypher_reference, encoding="utf-8")

cypher_path

WindowsPath('D:/_DS-Projects/Data-Science/databricks-learning-kg/cypher/03_initial_kg_queries.cypher')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 22. Relatório Markdown</strong></font>

<font size="2" color='#66666'></font></div>

In [27]:
report = f"""# Neo4j Load Summary v0.1 — Databricks Learning Knowledge Graph

## Projeto

{PROJECT_NAME}

## Carga no Neo4j

| Entidade | Total |
|---|---:|
| Nós carregados | {validation_df.loc[validation_df["check"] == "nodes", "total_in_neo4j"].iloc[0]} |
| Relacionamentos carregados | {validation_df.loc[validation_df["check"] == "relationships", "total_in_neo4j"].iloc[0]} |

## Nós por tipo

{nodes_by_label_neo4j.to_markdown(index=False)}

## Relacionamentos por tipo

{relationships_by_type_neo4j.to_markdown(index=False)}

## Tópicos por domínio

{topics_by_domain.to_markdown(index=False)}

## Pré-requisitos mais estruturantes

{prerequisite_hubs.to_markdown(index=False)}

## Leitura executiva

A carga inicial no Neo4j materializou o Knowledge Graph da jornada de certificação Databricks em uma base de grafos consultável.

A estrutura permite navegar da certificação para domínios, tópicos, subtópicos, notebooks, recursos, skills e snapshots de progresso.

As primeiras consultas já permitem identificar lacunas de evidência prática, tópicos prioritários e conceitos que funcionam como pré-requisitos para outros assuntos.
"""

report_path = DOCS_DIR / "neo4j_load_summary_v01.md"
report_path.write_text(report, encoding="utf-8")

report_path

WindowsPath('D:/_DS-Projects/Data-Science/databricks-learning-kg/docs/neo4j_load_summary_v01.md')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 23. Encerrar conexão</strong></font>

<font size="2" color='#66666'></font></div>

In [28]:
driver.close()

print("Conexão com Neo4j encerrada.")

Conexão com Neo4j encerrada.


In [29]:
%reload_ext watermark 
%watermark -a "Roberto-SSoares-LfLngLrnng" -d -t -u --iversions -v -m -h

Author: Roberto-SSoares-LfLngLrnng

Last updated: 2026-04-27 19:19:03

Python implementation: CPython
Python version       : 3.12.12
IPython version      : 9.13.0

Compiler    : MSC v.1944 64 bit (AMD64)
OS          : Windows
Release     : 11
Machine     : AMD64
Processor   : Intel64 Family 6 Model 158 Stepping 9, GenuineIntel
CPU cores   : 4
Architecture: 64bit

Hostname: PC-ROBERTO

dotenv: 0.9.9
neo4j : 6.1.0
pandas: 3.0.2
re    : 2.2.1



<div style="background-color:#f3f2ee">
    
<font size="6" color='#CC403E'><strong>Fim</strong></font>

<font size="2" color='#66666'></font></div>

In [30]:
#!uv pip install nbconvert -U -q
!jupyter nbconvert --to html --template-file my-template-html-v10.tpl 03_neo4j_load_and_cypher_queries.ipynb

[NbConvertApp] Converting notebook 03_neo4j_load_and_cypher_queries.ipynb to html
[NbConvertApp] Writing 62228 bytes to 03_neo4j_load_and_cypher_queries.html
